In [0]:
# dbutils.fs.ls("/Volumes/databrick_caresync/landing/landing_volume/neon_postgres/hospitals/2026-10-14_02:39:39/")

In [0]:
# Create widgets
dbutils.widgets.text("source_system", "", "Source System")
dbutils.widgets.text("table_name", "", "Table Name")
dbutils.widgets.text("landing_timestamp", "", "Landing Timestamp")
dbutils.widgets.text("source_file_format", "", "source_file_format")


# Get widget values into variables
source_system = dbutils.widgets.get("source_system")
table_name = dbutils.widgets.get("table_name")
landing_timestamp = dbutils.widgets.get("landing_timestamp")
source_file_format = dbutils.widgets.get("source_file_format")

In [0]:
if source_file_format == "csv":
    landing_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(f"/Volumes/databrick_caresync/landing/landing_volume/{source_system}/{table_name}/{landing_timestamp}/")
else:
    landing_df = spark.read.format("parquet").load(f"/Volumes/databrick_caresync/landing/landing_volume/{source_system}/{table_name}/{landing_timestamp}/")

In [0]:
#add two new dataframe

from pyspark.sql import functions as F

updated_df = landing_df.withColumn("landing_timestamp", F.lit(landing_timestamp))\
.withColumn("insert_timestamp",F.current_timestamp())


In [0]:
# display(updated_df)

In [0]:
#upend the data in bronze layer from landing
updated_df.write.mode("append").saveAsTable(f"databrick_caresync.bronze.{table_name}")